# DS-2002 Project 2 — Retail Sales Lakehouse

This builds on my midterm where I built a star schema for Montgomery County retail/warehouse 
sales in MySQL Workbench. For this project I'm adding streaming, multiple cloud sources, and 
the bronze/silver/gold pattern that the assignment asks for.

### Source layout
- **Aiven MySQL** holds dim_product and dim_supplier (relational source)
- **MongoDB Atlas** holds dim_item_type (NoSQL source)  
- **Databricks Volume** holds dim_date.csv and the 3 fact JSON files (file source)

### Notes on what changed from the original plan
The assignment suggested Azure SQL/MySQL, but UVA's Azure subscription wouldn't let me 
create resource groups, so I swapped to Aiven for MySQL — same JDBC pattern, same MySQL 
8 backend. I'm also using Databricks Free Trial (serverless), which means I can't run 
JDBC straight from the notebook because of egress restrictions. To work around that, I 
exported the MySQL dim tables to CSV and the Mongo collection to JSON, staged them in a 
Unity Catalog Volume, and read from there. Data still originates from those sources.

### Section I: Prerequisites

#### 1.0. Import Required Libraries

In [0]:
import os
import json
import pymongo
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.types import IntegerType, LongType, DoubleType

#### 2.0. Instantiate Global Variables

In [0]:
###connecting to mongodb
atlas_cluster_name  = "ds2002.wl5gd0p"
atlas_database_name = "retail_sales_dlh"
atlas_user_name     = "pvv5ag_db_user"
atlas_password      = "<my password>"

###connecting to databricks via Volume
volume_path = "/Volumes/workspace/retail_sales_dlh/source_data"

batch_dir   = volume_path
stream_dir  = f"{volume_path}/stream"

# Idempotent re-run cleanup (drop tables only, NOT volume contents)
spark.sql("DROP TABLE IF EXISTS retail_sales_dlh.fact_retail_sales_bronze")
spark.sql("DROP TABLE IF EXISTS retail_sales_dlh.fact_retail_sales_silver")

DataFrame[]

#### 3.0. Define Global Functions

In [0]:
### fetch documents from mongodb
def get_mongo_documents(user_id, pwd, cluster_name, db_name, collection):
    '''Create a client connection to MongoDB Atlas and return all documents from a collection.'''
    mongo_uri = f"mongodb+srv://{user_id}:{pwd}@{cluster_name}.mongodb.net/{db_name}?retryWrites=true&w=majority"
    client = pymongo.MongoClient(mongo_uri)
    db = client[db_name]
    docs = list(db[collection].find({}, {"_id": 0}))   ###drop Mongo internal id
    client.close()
    return docs


###json file in mongo
def set_mongo_collection(user_id, pwd, cluster_name, db_name, src_file_path, json_files):
    '''Create a client connection to MongoDB'''
    mongo_uri = f"mongodb+srv://{user_id}:{pwd}@{cluster_name}.mongodb.net/{db_name}?retryWrites=true&w=majority"
    client = pymongo.MongoClient(mongo_uri)
    db = client[db_name]
    
    '''Read in a JSON file, and Use It to Create a New Collection'''
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(src_file_path, json_files[file])
        with open(json_file, 'r') as openfile:
            json_object = json.load(openfile)
            file = db[file]
            result = file.insert_many(json_object)
    client.close()
    return result

### Section II: Populate Dimensions by Ingesting Reference (Cold-path) Data
#### 1.0. Create Databricks Metadata Database

In [0]:
###helpful for reruns
%sql
DROP TABLE IF EXISTS retail_sales_dlh.dim_product;
DROP TABLE IF EXISTS retail_sales_dlh.dim_supplier;
DROP TABLE IF EXISTS retail_sales_dlh.dim_item_type;
DROP TABLE IF EXISTS retail_sales_dlh.dim_date;
DROP TABLE IF EXISTS retail_sales_dlh.fact_retail_sales_bronze;
DROP TABLE IF EXISTS retail_sales_dlh.fact_retail_sales_silver;

In [0]:
###creating database
%sql
CREATE DATABASE IF NOT EXISTS retail_sales_dlh

#### 2.0. Load `dim_product` (Originates from Aiven MySQL — staged via CSV)

In [0]:
###bring in dim_product from Aiven (~21k diff products)
df_product = (
    spark.read.format("csv")
         .options(header="true", inferSchema="true")
         .load(f"{batch_dir}/dim_product_from_mysql.csv")
)
df_product.write.format("delta").mode("overwrite").saveAsTable("retail_sales_dlh.dim_product")
display(df_product.limit(5))
print(f"dim_product: {df_product.count():,} rows")

product_key,item_code,item_description,item_type
1,100009,BOOTLEG RED - 750ML,WINE
2,100024,MOMENT DE PLAISIR - 750ML,WINE
3,1001,S SMITH ORGANIC PEAR CIDER - 18.7OZ,BEER
4,100145,SCHLINK HAUS KABINETT - 750ML,WINE
5,100293,SANTORINI GAVALA WHITE - 750ML,WINE


dim_product: 34,056 rows


#### 3.0. Load `dim_supplier` (Originates from Aiven MySQL — staged via CSV)

In [0]:
###suppliers now (~400)
df_supplier = (
    spark.read.format("csv")
         .options(header="true", inferSchema="true")
         .load(f"{batch_dir}/dim_supplier_from_mysql.csv")
)
df_supplier.write.format("delta").mode("overwrite").saveAsTable("retail_sales_dlh.dim_supplier")
display(df_supplier.limit(5))
print(f"dim_supplier: {df_supplier.count():,} rows")

supplier_key,supplier_name
1,REPUBLIC NATIONAL DISTRIBUTING CO
2,PWSWN INC
3,RELIABLE CHURCHILL LLLP
4,LANTERNA DISTRIBUTORS INC
5,DIONYSOS IMPORTS INC


dim_supplier: 396 rows


#### 4.0. Load `dim_item_type` from MongoDB Atlas (NoSQL Source)

In [0]:
###load dim_item_type
df_item_type = (
    spark.read.format("json")
         .option("multiLine", "true")
         .load(f"{batch_dir}/dim_item_type.json")
)
df_item_type.write.format("delta").mode("overwrite").saveAsTable("retail_sales_dlh.dim_item_type")
display(df_item_type)
print(f"dim_item_type: {df_item_type.count()} rows")

item_type,item_type_key
WINE,1
BEER,2
LIQUOR,3
STR_SUPPLIES,4
KEGS,5
REF,6
DUNNAGE,7
NON-ALCOHOL,8
WINE,1
BEER,2


dim_item_type: 9 rows


#### 5.0. Load `dim_date` from DBFS CSV (File System Source)

In [0]:
###bring in dim_date from csv
df_date = (
    spark.read.format("csv")
         .options(header="true", inferSchema="true")
         .load(f"{batch_dir}/dim_date.csv")
)
df_date.write.format("delta").mode("overwrite").saveAsTable("retail_sales_dlh.dim_date")
display(df_date.limit(5))
print(f"dim_date: {df_date.count():,} rows")

date_key,full_date
20170601,2017-06-01
20170701,2017-07-01
20170801,2017-08-01
20170901,2017-09-01
20171001,2017-10-01


dim_date: 24 rows


##### Verify all dimension tables

In [0]:
###check to see if my volumes worked in databricks
%sql
USE retail_sales_dlh;
SHOW TABLES

database,tableName,isTemporary
retail_sales_dlh,dim_date,false
retail_sales_dlh,dim_item_type,false
retail_sales_dlh,dim_product,false
retail_sales_dlh,dim_supplier,false


### Section III: Integrate Reference Data with Real-Time Data
#### 6.0. Bronze Table: Stream Raw JSON Data via Structured Streaming

We use the standard `spark.readStream` pattern with `maxFilesPerTrigger=1` so each fact JSON file is ingested as a separate streaming interval, simulating real-time arrival of three batches.

In [0]:
###bronze: raw streamed rows + ingestion metadata
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, DoubleType
from pyspark.sql.functions import current_timestamp, col

fact_schema = StructType([
    StructField("date_key",         LongType(),    False),
    StructField("product_key",      IntegerType(), False),
    StructField("supplier_key",     IntegerType(), False),
    StructField("item_type_key",    IntegerType(), False),
    StructField("retail_sales",     DoubleType(),  True),
    StructField("retail_transfers", DoubleType(),  True),
    StructField("warehouse_sales",  DoubleType(),  True),
])

(spark.readStream
    .schema(fact_schema)
    .option("maxFilesPerTrigger", 1)
    .json(stream_dir)
    .withColumn("receipt_time", current_timestamp())
    .withColumn("source_file",  col("_metadata.file_path"))
    .createOrReplaceTempView("sales_bronze_tempview"))

In [0]:
(spark.table("sales_bronze_tempview")
      .writeStream
      .format("delta")
      .option("checkpointLocation", f"{volume_path}/_checkpoints/bronze")
      .outputMode("append")
      .trigger(availableNow=True)
      .toTable("retail_sales_dlh.fact_retail_sales_bronze")
      .awaitTermination())

#### 6.2. Silver Table: Join Streaming Fact with Reference Dimensions

In [0]:
###silver: bronze joined to all 4 dims
(spark.readStream
      .table("retail_sales_dlh.fact_retail_sales_bronze")
      .createOrReplaceTempView("sales_silver_tempview"))

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW fact_retail_sales_silver_tempview AS (
  SELECT t.date_key
    , d.full_date
    , t.product_key
    , p.item_code
    , p.item_description
    , t.supplier_key
    , s.supplier_name
    , t.item_type_key
    , it.item_type
    , t.retail_sales
    , t.retail_transfers
    , t.warehouse_sales
    , (t.retail_sales + t.retail_transfers + t.warehouse_sales) AS total_movement
    , t.receipt_time
    , t.source_file
  FROM sales_silver_tempview t
  INNER JOIN retail_sales_dlh.dim_product p
  ON t.product_key = p.product_key
  INNER JOIN retail_sales_dlh.dim_supplier s
  ON t.supplier_key = s.supplier_key
  INNER JOIN retail_sales_dlh.dim_item_type it
  ON t.item_type_key = it.item_type_key
  INNER JOIN retail_sales_dlh.dim_date d
  ON t.date_key = d.date_key
)

In [0]:
(spark.table("fact_retail_sales_silver_tempview")
      .writeStream
      .format("delta")
      .option("checkpointLocation", f"{volume_path}/_checkpoints/silver")
      .outputMode("append")
      .trigger(availableNow=True)
      .toTable("retail_sales_dlh.fact_retail_sales_silver")
      .awaitTermination())

In [0]:
%sql
SELECT * FROM retail_sales_dlh.fact_retail_sales_silver LIMIT 10

date_key,full_date,product_key,item_code,item_description,supplier_key,supplier_name,item_type_key,item_type,retail_sales,retail_transfers,warehouse_sales,total_movement,receipt_time,source_file
20170601,2017-06-01,7695,47155,CLYDES CALIF CAB - 750ML,246,RUTHERFORD WINE COMPANY,1,WINE,0.0,0.5,23.0,23.5,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json
20170601,2017-06-01,19892,45144,H WALKER BRANDY - BLACKBERRY - 200ML,23,PERNOD RICARD USA LLC,3,LIQUOR,0.4,0.0,0.0,0.4,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json
20170601,2017-06-01,19893,45154,DIMPLE PINCH SCOTCH 15YR - 1.75L,43,DIAGEO NORTH AMERICA INC,3,LIQUOR,0.84,1.0,0.0,1.8399999999999999,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json
20170601,2017-06-01,7547,45160,LILLET ROUGE - 750ML,23,PERNOD RICARD USA LLC,1,WINE,1.69,1.0,-1.0,1.69,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json
20170601,2017-06-01,7548,45163,LOUIS LATOUR GRD ARDECHE CHARD - 750ML,13,MONSIEUR TOUTON SELECTION,1,WINE,2.57,2.0,1.0,5.57,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json
20170601,2017-06-01,7549,45175,ARDBEG CORRYVRECKAN - 750ML,107,MOET HENNESSY USA,3,LIQUOR,6.77,3.0,0.0,9.77,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json
20170601,2017-06-01,19894,45177,REDTREE P/NOIR - 750ML,44,OPICI FAMILY DISTRIBUTING OF MD,1,WINE,0.0,0.0,3.0,3.0,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json
20170601,2017-06-01,7550,45184,REMY MARTIN XO EXCELLENCE - 750ML,104,REMY COINTREAU USA,3,LIQUOR,1.61,0.0,0.0,1.61,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json
20170601,2017-06-01,13978,45214,MCMANIS FAMILY VYDS PET/SHZ - 750ML,44,OPICI FAMILY DISTRIBUTING OF MD,1,WINE,0.5,0.0,1.0,1.5,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json
20170601,2017-06-01,7551,45218,MARS & VENUS MER - 750ML,288,A VINTNERS SELECTIONS,1,WINE,1.05,3.0,1.0,5.05,2026-05-08T22:07:40.616Z,dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json


#### 6.4. Gold Table: Business-Value Aggregations

In [0]:
###Gold: business-value queries

%sql
###1: top 20 suppliers by movement of product
SELECT supplier_name
  , ROUND(SUM(retail_sales),    2) AS total_retail_sales
  , ROUND(SUM(warehouse_sales), 2) AS total_warehouse_sales
  , ROUND(SUM(total_movement),  2) AS total_movement
  , COUNT(DISTINCT product_key)    AS distinct_products
FROM retail_sales_dlh.fact_retail_sales_silver
GROUP BY supplier_name
ORDER BY total_movement DESC
LIMIT 20

supplier_name,total_retail_sales,total_warehouse_sales,total_movement,distinct_products
CROWN IMPORTS,168875.32,3303743.02,3638283.86,67
MILLER BREWING COMPANY,174312.88,2850857.42,3195617.88,198
ANHEUSER BUSCH INC,219921.64,2662341.68,3098725.36,427
HEINEKEN USA,112279.82,1659592.92,1881593.58,70
E & J GALLO WINERY,332341.06,394927.56,1057312.26,617
DIAGEO NORTH AMERICA INC,290686.4,341129.26,919843.08,481
CONSTELLATION BRANDS,263329.58,238799.02,761652.2,375
BOSTON BEER CORPORATION,80258.26,383185.78,543650.46,186
THE WINE GROUP,128540.08,144383.14,400633.28,232
JIM BEAM BRANDS CO,192328.08,15885.62,398672.44,346


In [0]:
%sql
--2: Top 10 products by retail sales
SELECT item_code
  , item_description
  , item_type
  , ROUND(SUM(retail_sales),    2) AS total_retail_sales
  , ROUND(SUM(warehouse_sales), 2) AS total_warehouse_sales
  , COUNT(*)                       AS transaction_count
FROM retail_sales_dlh.fact_retail_sales_silver
GROUP BY item_code, item_description, item_type
ORDER BY total_retail_sales DESC
LIMIT 10

item_code,item_description,item_type,total_retail_sales,total_warehouse_sales,transaction_count
53929,TITO'S HANDMADE VODKA - 1.75L,LIQUOR,55161.0,973.66,48
23445,CORONA EXTRA LOOSE NR - 12OZ,BEER,50128.0,606321.66,48
23886,HEINEKEN LOOSE NR - 12OZ,BEER,35522.0,343900.42,48
90590,MILLER LITE 30PK CAN - 12OZ,BEER,28880.0,268972.8,48
90468,BUD LIGHT 30PK CAN,BEER,24598.0,192633.94,48
35840,BOWMAN'S VODKA - 1.75L,LIQUOR,24253.44,328.0,48
23905,MILLER LITE HIGH GRAPHIC LOOSE NR - 12OZ,BEER,20747.54,51264.68,48
26187,STELLA ARTOIS LOOSE NR - 11.2OZ,BEER,20708.0,45779.84,48
96750,CORONA EXTRA 2/12 NR - 12OZ,BEER,19510.38,495848.86,48
40932,PINNACLE VODKA - 1.75L,LIQUOR,17352.8,148.0,48


In [0]:
%sql
--3: Sales by item type per period
SELECT date_key
  , full_date
  , item_type
  , ROUND(SUM(retail_sales),    2) AS total_retail_sales
  , ROUND(SUM(warehouse_sales), 2) AS total_warehouse_sales
  , ROUND(SUM(total_movement),  2) AS total_movement
  , COUNT(*)                       AS transaction_count
FROM retail_sales_dlh.fact_retail_sales_silver
GROUP BY date_key, full_date, item_type
ORDER BY date_key, total_movement DESC

date_key,full_date,item_type,total_retail_sales,total_warehouse_sales,total_movement,transaction_count
20170601,2017-06-01,BEER,55726.52,652385.88,765841.24,3520
20170601,2017-06-01,WINE,66259.28,95402.92,223537.82,16714
20170601,2017-06-01,LIQUOR,70070.62,8733.36,145403.7,5870
20170601,2017-06-01,KEGS,0.0,12472.0,12472.0,968
20170601,2017-06-01,NON-ALCOHOL,2418.46,2687.5,7638.3,142
20170601,2017-06-01,STR_SUPPLIES,178.44,0.0,783.92,24
20170601,2017-06-01,REF,61.2,0.0,159.2,6
20170601,2017-06-01,DUNNAGE,0.0,-12750.0,-12750.0,6
20170701,2017-07-01,BEER,54314.22,542815.74,647220.7,3368
20170701,2017-07-01,WINE,61461.18,83421.82,204189.38,15844


In [0]:
%sql
-- Gold Query 4: Confirm streaming intervals were ingested
SELECT source_file, COUNT(*) AS row_count
FROM retail_sales_dlh.fact_retail_sales_silver
GROUP BY source_file
ORDER BY source_file

source_file,row_count
dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_01.json,219209
dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_02.json,201824
dbfs:/Volumes/workspace/retail_sales_dlh/source_data/stream/fact_retail_sales_03.json,193922
